# Cargar `datos_clusterizados.csv` en MySQL

Este notebook lee `data/datos_clusterizados.csv` (separado por punto y coma), valida sus columnas y carga los 21 campos en la tabla `medicamentos_clusterizados` de MySQL. Al volver a ejecutarlo, sustituye las filas de esa tabla por el contenido actual del CSV; no modifica otras tablas.


## 1. Instalar dependencias

Ejecuta esta celda una vez en el kernel del notebook.

In [42]:
%pip install pandas mysql-connector-python

/Users/santiago/Desktop/Data Engineering/Proyecto-Ingenieria-Dato/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [43]:
%pip install mysql-connector-python

/Users/santiago/Desktop/Data Engineering/Proyecto-Ingenieria-Dato/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [44]:
from getpass import getpass
from pathlib import Path

import mysql.connector
import numpy as np
import pandas as pd

## 2. Leer el CSV

La ruta predeterminada funciona al ejecutar el notebook desde la raíz del proyecto o desde `Notebooks/`. Si el archivo está en otro sitio, actualiza `CSV_PATH`.

In [45]:
candidate_paths = [
    Path("data/datos_clusterizados.csv"),
    Path("../data/datos_clusterizados.csv"),
]
CSV_PATH = next((path for path in candidate_paths if path.is_file()), candidate_paths[0])

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"No se encontró el CSV. Comprueba CSV_PATH: {CSV_PATH.resolve()}")

df = pd.read_csv(CSV_PATH, sep=";", encoding="utf-8")
print(f"Archivo: {CSV_PATH.resolve()}")
print(f"Filas y columnas: {df.shape}")
display(df.head())

Archivo: /Users/santiago/Desktop/Data Engineering/Proyecto-Ingenieria-Dato/data/datos_clusterizados.csv
Filas y columnas: (70, 21)


,nombre,pactivos,labtitular,labcomercializador,dosis (mg),forma_farmaceutica_simplificada,estado_aut,estado_rev,vias_administracion,comercializado,...,afecta_conduccion,num_principios_activos,num_excipientes,Volumen_Informativo,Complejidad_Desglose,Indicador_Riesgo,Estado,Precio venta al público con IVA,Tratamiento de larga duración,cluster
0,"ALMOTRIPTAN CINFA 12,5 MG COMPRIMIDOS RECUBIER...",ALMOTRIPTAN,Laboratorios Cinfa S.A.,Laboratorios Cinfa S.A.,12.5,COMPRIMIDO,2014,2014,VÍA ORAL,1,...,1,1,3,659,1,5,ALTA,19.53,0,1
1,"ALMOTRIPTAN NORMON 12,5 MG COMPRIMIDOS RECUBIE...",ALMOTRIPTAN,Laboratorios Normon S.A.,Laboratorios Normon S.A.,12.5,COMPRIMIDO,2013,2013,VÍA ORAL,1,...,1,1,3,678,1,5,ALTA,19.53,0,1
2,"ALMOTRIPTAN STADA 12,5 MG COMPRIMIDOS RECUBIER...",ALMOTRIPTAN,Laboratorio Stada S.L.,Laboratorio Stada S.L.,12.5,COMPRIMIDO,2014,2014,VÍA ORAL,1,...,1,1,4,666,1,5,ALTA,19.53,0,1
3,IBUPROFENO (ARGININA) KERN PHARMA 600 mg GRANU...,IBUPROFENO,Kern Pharma S.L.,Kern Pharma S.L.,600.0,SOLUCIÓN,2008,2008,VÍA ORAL,1,...,0,1,5,1432,0,4,BAJA POR NO COMERCIALIZAR,2.50,0,3
4,IBUPROFENO STADA 600 MG COMPRIMIDOS RECUBIERTO...,IBUPROFENO,Laboratorio Stada S.L.,Laboratorio Stada S.L.,600.0,COMPRIMIDO,2013,2013,VÍA ORAL,1,...,0,1,2,1279,0,4,ALTA,1.97,0,5


## 3. Preparar y validar las columnas

Se renombran las columnas con espacios, tildes o paréntesis para usar identificadores sencillos en SQL. Se conserva la información de todas las columnas del CSV.

In [46]:
column_mapping = {
    "nombre": "nombre",
    "pactivos": "principios_activos",
    "labtitular": "laboratorio_titular",
    "labcomercializador": "laboratorio_comercializador",
    "dosis (mg)": "dosis_mg",
    "forma_farmaceutica_simplificada": "forma_farmaceutica",
    "estado_aut": "estado_autorizacion",
    "estado_rev": "estado_revision",
    "vias_administracion": "via_administracion",
    "comercializado": "comercializado",
    "generico": "generico",
    "afecta_conduccion": "afecta_conduccion",
    "num_principios_activos": "num_principios_activos",
    "num_excipientes": "num_excipientes",
    "Volumen_Informativo": "volumen_informativo",
    "Complejidad_Desglose": "complejidad_desglose",
    "Indicador_Riesgo": "indicador_riesgo",
    "Estado": "estado",
    "Precio venta al público con IVA": "precio_venta_publico_iva",
    "Tratamiento de larga duración": "tratamiento_larga_duracion",
    "cluster": "cluster",
}

missing_columns = [column for column in column_mapping if column not in df.columns]
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas en el CSV: {missing_columns}")
if df.empty:
    raise ValueError("El CSV no contiene filas para cargar.")

# Verifica que los campos numéricos se puedan convertir y no tengan valores vacíos.
numeric_columns = [
    "dosis (mg)", "estado_aut", "estado_rev", "comercializado", "generico",
    "afecta_conduccion", "num_principios_activos", "num_excipientes",
    "Volumen_Informativo", "Complejidad_Desglose", "Indicador_Riesgo",
    "Precio venta al público con IVA", "Tratamiento de larga duración", "cluster",
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="raise")
if df[list(column_mapping)].isna().any().any():
    raise ValueError("El CSV contiene valores vacíos en columnas que la tabla requiere.")

df_load = df.rename(columns=column_mapping).loc[:, list(column_mapping.values())].copy()
print(f"Filas validadas para cargar: {len(df_load)}")
display(df_load.head())

Filas validadas para cargar: 70


,nombre,principios_activos,laboratorio_titular,laboratorio_comercializador,dosis_mg,forma_farmaceutica,estado_autorizacion,estado_revision,via_administracion,comercializado,...,afecta_conduccion,num_principios_activos,num_excipientes,volumen_informativo,complejidad_desglose,indicador_riesgo,estado,precio_venta_publico_iva,tratamiento_larga_duracion,cluster
0,"ALMOTRIPTAN CINFA 12,5 MG COMPRIMIDOS RECUBIER...",ALMOTRIPTAN,Laboratorios Cinfa S.A.,Laboratorios Cinfa S.A.,12.5,COMPRIMIDO,2014,2014,VÍA ORAL,1,...,1,1,3,659,1,5,ALTA,19.53,0,1
1,"ALMOTRIPTAN NORMON 12,5 MG COMPRIMIDOS RECUBIE...",ALMOTRIPTAN,Laboratorios Normon S.A.,Laboratorios Normon S.A.,12.5,COMPRIMIDO,2013,2013,VÍA ORAL,1,...,1,1,3,678,1,5,ALTA,19.53,0,1
2,"ALMOTRIPTAN STADA 12,5 MG COMPRIMIDOS RECUBIER...",ALMOTRIPTAN,Laboratorio Stada S.L.,Laboratorio Stada S.L.,12.5,COMPRIMIDO,2014,2014,VÍA ORAL,1,...,1,1,4,666,1,5,ALTA,19.53,0,1
3,IBUPROFENO (ARGININA) KERN PHARMA 600 mg GRANU...,IBUPROFENO,Kern Pharma S.L.,Kern Pharma S.L.,600.0,SOLUCIÓN,2008,2008,VÍA ORAL,1,...,0,1,5,1432,0,4,BAJA POR NO COMERCIALIZAR,2.50,0,3
4,IBUPROFENO STADA 600 MG COMPRIMIDOS RECUBIERTO...,IBUPROFENO,Laboratorio Stada S.L.,Laboratorio Stada S.L.,600.0,COMPRIMIDO,2013,2013,VÍA ORAL,1,...,0,1,2,1279,0,4,ALTA,1.97,0,5


## 4. Definir la tabla y la inserción

La tabla incluye un `id` autoincremental y todas las columnas del archivo. Los tipos están ajustados al contenido del CSV; `utf8mb4` permite guardar texto con tildes.

In [47]:
TABLE_NAME = "medicamentos_clusterizados"

create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT PRIMARY KEY,
    nombre VARCHAR(512) NOT NULL,
    principios_activos VARCHAR(255) NOT NULL,
    laboratorio_titular VARCHAR(255) NOT NULL,
    laboratorio_comercializador VARCHAR(255) NOT NULL,
    dosis_mg DECIMAL(10, 2) NOT NULL,
    forma_farmaceutica VARCHAR(150) NOT NULL,
    estado_autorizacion SMALLINT NOT NULL,
    estado_revision SMALLINT NOT NULL,
    via_administracion VARCHAR(150) NOT NULL,
    comercializado TINYINT NOT NULL,
    generico TINYINT NOT NULL,
    afecta_conduccion TINYINT NOT NULL,
    num_principios_activos SMALLINT NOT NULL,
    num_excipientes SMALLINT NOT NULL,
    volumen_informativo INT NOT NULL,
    complejidad_desglose SMALLINT NOT NULL,
    indicador_riesgo SMALLINT NOT NULL,
    estado VARCHAR(100) NOT NULL,
    precio_venta_publico_iva DECIMAL(10, 2) NOT NULL,
    tratamiento_larga_duracion TINYINT NOT NULL,
    cluster INT NOT NULL
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""

columns = list(df_load.columns)
quoted_columns = ", ".join(f"`{column}`" for column in columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_sql = f"INSERT INTO {TABLE_NAME} ({quoted_columns}) VALUES ({placeholders})"

## 5. Conectar y cargar los datos

Actualiza el usuario/base de datos si es necesario. La contraseña se solicita de forma interactiva y no se guarda en el notebook. La carga borra y repone **solo** `medicamentos_clusterizados`; `DELETE` y los inserts se confirman juntos, y ante un error se revierten.

In [49]:
config = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "database": "MUIAAp_2026",
    "charset": "utf8mb4",
    "connection_timeout": 10,
    "autocommit": False,
}

# Convierte escalares de NumPy a tipos nativos que acepta mysql-connector.
rows = [
    tuple(value.item() if isinstance(value, np.generic) else value for value in row)
    for row in df_load.itertuples(index=False, name=None)
]

conn = None
cursor = None
try:
    conn = mysql.connector.connect(password=getpass("MySQL password: "), **config)
    cursor = conn.cursor()
    cursor.execute(create_table_sql)
    cursor.execute(f"DELETE FROM {TABLE_NAME}")
    cursor.executemany(insert_sql, rows)

    cursor.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}")
    sql_count = cursor.fetchone()[0]
    if sql_count != len(df_load):
        raise ValueError(f"El recuento no coincide: SQL={sql_count}, CSV={len(df_load)}")

    conn.commit()
    print(f"Carga confirmada: {sql_count} filas en {config['database']}.{TABLE_NAME}")
except Exception:
    if conn is not None and conn.is_connected():
        conn.rollback()
    raise
finally:
    if cursor is not None:
        cursor.close()
    if conn is not None and conn.is_connected():
        conn.close()

Carga confirmada: 70 filas en MUIAAp_2026.medicamentos_clusterizados


## 6. Verificar los datos en MySQL

Puedes ejecutar estas consultas en MySQL Workbench para revisar la estructura, el total de filas y la distribución de medicamentos por clúster.

In [ ]:
print(f"Filas del CSV: {len(df_load)}")
print("Filas por clúster en el CSV:")
display(df_load.groupby("cluster").size().rename("filas"))

```sql
USE MUIAAp_2026;

SELECT * FROM medicamentos_clusterizados;

SELECT volumen_informativo / complejidad_desglose AS resultado_division 
FROM medicamentos_clusterizados;
```